# Assignment: MLflow Tracking with House Prices Dataset

In [ ]:
# 1. Setup
# !pip install mlflow scikit-learn pandas matplotlib seaborn  # Uncomment to install

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import mlflow
import mlflow.sklearn

print("Libraries imported successfully.")

## 2. Load Data

In [ ]:
# Load the dataset
df = pd.read_csv('train.csv')

# Display the first few rows
print(df.head())
print(f"Dataset shape: {df.shape}")

## 3. Preprocessing

In [ ]:
# Select numeric features as predictors
features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'FullBath', 'YearBuilt']
target = 'SalePrice'

# Handle missing values by filling with median
X = df[features].copy()
y = df[target]

# Log missing values count (Optional part)
missing_counts = X.isnull().sum().sum()
print(f"Total missing values handled: {missing_counts}")
X = X.fillna(X.median())

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Preprocessing complete.")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 4. MLflow Basics

In [ ]:
# Check the tracking URI
print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")

# Note: To start the UI, run 'mlflow ui --port 5000' in your terminal and visit http://localhost:5000

## 5. Train and Log Experiments

In [ ]:
def train_and_log_rf(n_estimators, max_depth):
    with mlflow.start_run(run_name=f"RF_est{n_estimators}_depth{max_depth}"):
        # Train model
        model = RandomForestRegressor(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)
        
        # Predict and calculate metrics
        predictions = model.predict(X_test)
        mse = mean_squared_error(y_test, predictions)
        rmse = np.sqrt(mse)
        
        # Log parameters
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("missing_values_handled", missing_counts)
        
        # Log metrics
        mlflow.log_metric("mse", mse)
        mlflow.log_metric("rmse", rmse)
        
        # Log model artifact
        mlflow.sklearn.log_model(model, "model")
        
        # Feature Importance Plot (Optional)
        importance = pd.Series(model.feature_importances_, index=features).sort_values()
        plt.figure(figsize=(10, 6))
        importance.plot(kind='barh')
        plt.title('Feature Importance')
        plt.savefig("importance.png")
        plt.close()
        mlflow.log_artifact("importance.png")
        
        print(f"Run completed: RMSE={rmse:.2f}")

# Execute training with different parameters
params_list = [
    (10, 5),
    (50, 10),
    (100, 20)
]

for n_est, depth in params_list:
    train_and_log_rf(n_est, depth)

## 6. Extension: Different Model (Linear Regression)

In [ ]:
with mlflow.start_run(run_name="Linear_Regression"):
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    
    preds = lr.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    
    mlflow.log_metric("rmse", rmse)
    mlflow.sklearn.log_model(lr, "model")
    
    print(f"Linear Regression RMSE: {rmse:.2f}")

## 7. Compare Runs
Check the MLflow UI (http://localhost:5000) to see the comparisons, sort by RMSE, and view artifacts.